RunnableBranch
- 입력에 따라 동적으로 로직을 라우팅할 수 있는 도구. 입력 데이터의 특성에 기반해서 다양한 처리 경로를 유연하게 정의 가능.
- 복잡한 의사 결정 트리를 간단하고 직관적으로 구현 가능
- 런타임에 동적으로 분기 조건을 평가하고 적절한 처리 루틴 선택 가능

In [ ]:
from dotenv import load_dotenv
from operator import itemgetter

from langchain_teddynote import logging
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

from langchain_core.runnables import RunnableLambda, RunnableBranch

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-LCEL-Advanced")

수학, 과학, 기타 중 하나로 분류하는 chain

In [ ]:
prompt = PromptTemplate.from_template(
    """주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요. 한 단어 이상으로 응답하지 마세요.

<question>
{question}
</question>

Classification:"""
)

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
chain = (prompt | llm | StrOutputParser())

In [ ]:
chain.invoke({"question": "2+2 는 무엇인가요?"})

In [ ]:
chain.invoke({"question": "작용 반작용의 법칙은 무엇인가요?"})

In [ ]:
chain.invoke({"question": "Google은 어떤 회사인가요?"})

In [ ]:
math_prompt = PromptTemplate.from_template(
        """You are an expert in math. \
Always answer questions starting with "깨봉선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
)

science_prompt = PromptTemplate.from_template(
        """You are an expert in science. \
Always answer questions starting with "아이작 뉴턴 선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
)

general_prompt = PromptTemplate.from_template(
        """Respond to the following question concisely:

Question: {question}
Answer:"""
)

In [ ]:
math_chain = (math_prompt | llm)
science_chain = (science_prompt | llm)
general_chain = (general_prompt | llm)

사용자 정의 함수로 라우팅 + RunnableLambda

In [ ]:
def route(info):
    if "수학" in info["topic"].lower():  # 주제에 "수학"이 포함되어 있는 경우
        return math_chain
    elif "과학" in info["topic"].lower():  # 주제에 "과학"이 포함되어 있는 경우
        return science_chain
    else:  # 그 외의 경우
        return general_chain

In [ ]:
full_chain1 = (
    {"topic": chain, "question": itemgetter("question")} 
    | RunnableLambda(route)  # 경로를 지정하는 함수를 인자로 전달
    | StrOutputParser()
)

In [ ]:
full_chain1.invoke({"question": "미적분의 개념에 대해 말씀해 주세요."})

In [ ]:
full_chain1.invoke({"question": "중력은 어떻게 작용하나요?"})

In [ ]:
full_chain1.invoke({"question": "RAG(Retrieval Augmented Generation)은 무엇인가요?"})

RunnableBranch

In [ ]:
branch = RunnableBranch(
    (lambda x: "수학" in x["topic"].lower(), math_chain),  # 주제에 "수학" 용어가 포함되면 수학 체인
    (lambda x: "과학" in x["topic"].lower(), science_chain),  # 주제에 "과학" 용어가 포함되면 과학 체인
    general_chain  # 위의 조건 중 어느것도 해당하지 않으면 기타
)

In [ ]:
full_chain2 = ({"topic": chain, "question": itemgetter("question")} | branch | StrOutputParser())

In [ ]:
full_chain2.invoke({"question": "미적분의 개념에 대해 말씀해 주세요."})

In [ ]:
full_chain2.invoke({"question": "중력 가속도는 어떻게 계산하나요?"})

In [ ]:
full_chain2.invoke({"question": "RAG(Retrieval Augmented Generation)은 무엇인가요?"})